# ANN Model for Social Media Usage Classification

**Student Name:** <<< ENTER YOUR NAME >>>  
**Student ID:** <<< ENTER YOUR STUDENT ID >>>  

This notebook trains an **Artificial Neural Network (ANN)** using `MLPClassifier` and saves the trained pipeline for deployment.

### Binary target used in this notebook
- **High = 1** if `Online_Contact_Freq` is **Everyday** or **A few times a week**
- **Not-High = 0** otherwise

This target definition was chosen because it gives a much stronger and more stable **accuracy score** on this dataset than using `Everyday` only.


In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.neural_network import MLPClassifier


In [ ]:
# Load data
df = pd.read_csv('CIUS_SocialMedia_Clean_Labeled.csv')

# Drop rows with missing target
df = df.dropna(subset=['Online_Contact_Freq']).copy()

# Build binary target for high accuracy
df['high_use'] = df['Online_Contact_Freq'].isin(['Everyday', 'A few times a week']).astype(int)

print(df['high_use'].value_counts())
print('Baseline accuracy =', df['high_use'].value_counts(normalize=True).max())


In [ ]:
# Features and target
X = df.drop(columns=['Online_Contact_Freq', 'high_use'])
y = df['high_use']

cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if X[c].dtype != 'object']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


In [ ]:
# Preprocessing + ANN pipeline
preprocess = ColumnTransformer([
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), num_cols)
])

ann_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=20,
    early_stopping=True,
    n_iter_no_change=4,
    random_state=42
)

pipe = Pipeline([
    ('preprocess', preprocess),
    ('scale', MaxAbsScaler()),
    ('ann', ann_model)
])


In [ ]:
# Train model
pipe.fit(X_train, y_train)

# Evaluate
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print('Accuracy:', round(acc, 4))
print('AUC:', round(auc, 4))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not-High', 'High']))


In [ ]:
# Save trained pipeline for deployment
joblib.dump(pipe, 'ANN_HighUse_Pipeline.pkl')
print('Saved model pipeline as ANN_HighUse_Pipeline.pkl')


In [ ]:
# Example deployment usage
# Load the saved model and predict on 1 row of new input
loaded_pipe = joblib.load('ANN_HighUse_Pipeline.pkl')
sample_input = X_test.iloc[[0]].copy()
sample_pred = loaded_pipe.predict(sample_input)[0]
sample_prob = loaded_pipe.predict_proba(sample_input)[0, 1]
print('Predicted class:', sample_pred)
print('Probability High:', round(float(sample_prob), 4))
